# Sofa vs table image classifier

A small computer vision project built while following lesson 1 of [Practical Deep Learning for Coders](https://course.fast.ai/) (fast.ai).

**What it does:** takes a photo and decides whether it shows a **sofa** or a **table**.

**How it works:**
1. Download photos for each class from DuckDuckGo image search (33 usable images per class in my run, 66 in total).
2. Fine-tune a pretrained `resnet18` network with `fastai` on these photos (80% training / 20% validation).
3. Test the model on photos it has never seen.

**Result:** about 85% accuracy on the validation set; see the Results section at the end.

**Stack:** Python, fastai, PyTorch, Kaggle notebooks (free GPU).

In [ ]:
# Kaggle needs a verified phone number to enable the internet and the GPU.
# This cell only checks that the internet is on.
import socket
try:
    socket.setdefaulttimeout(1)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('1.1.1.1', 53))
except socket.error:
    raise Exception("No internet. Turn the 'Internet' switch on in the notebook settings.")

In [ ]:
# Install / upgrade the libraries (pip warnings about root or incompatibilities can be ignored)
!pip install -Uqq fastai ddgs

## Step 1: Download the images

In [ ]:
import time
from ddgs import DDGS
from fastcore.all import *
from fastdownload import download_url
from fastai.vision.all import *

def search_images(keywords, max_images=200):
    """Return a list of image URLs found by DuckDuckGo for the given keywords."""
    return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')

First a quick check that the search works: download one sofa photo and look at it.

*Note: DuckDuckGo sometimes rate-limits requests. If you get an error, wait a minute and run the cell again.*

In [ ]:
urls = search_images('sofa', max_images=1)
download_url(urls[0], 'sofa.jpg', show_progress=False)
Image.open('sofa.jpg').to_thumb(256, 256)

In [ ]:
# Same for a table
download_url(search_images('table', max_images=1)[0], 'table.jpg', show_progress=False)
Image.open('table.jpg').to_thumb(256, 256)

The searches give reasonable results, so now download the photos for each class into its own folder (the folder name becomes the label) and shrink them to save memory. The search can return up to 200 results, but in my run only 33 images per class ended up in the folders (probably because some links were broken or the downloads timed out):

In [ ]:
searches = 'table', 'sofa'
path = Path('sofa_or_table')

for o in searches:
    dest = path/o
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo'))
    time.sleep(5)  # be polite to the search engine
    resize_images(path/o, max_size=400, dest=path/o)

## Step 2: Train the model

Some downloads may be broken files, which would crash training, so remove them first:

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
print(f'Removed {len(failed)} broken images')

In [ ]:
# How many images do we really have per class?
for o in searches:
    print(o, len(get_image_files(path/o)))

Create the `DataLoaders`: a *training set* (images used to learn) and a *validation set* (images used only to measure accuracy).

- `blocks=(ImageBlock, CategoryBlock)`: inputs are images, outputs are categories (`sofa` or `table`)
- `get_items=get_image_files`: find all image files in the folder
- `splitter=RandomSplitter(valid_pct=0.2, seed=42)`: random 80/20 split
- `get_y=parent_label`: the label is the name of the folder the image is in
- `item_tfms=[Resize(192, method='squish')]`: resize every image to 192x192 pixels

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

Now train. I use `resnet18`, a small and fast pretrained network. *Fine-tuning* means starting from a model that was already trained on ImageNet (1000 categories of everyday photos) and adjusting it a little so it learns to tell sofas from tables. `error_rate` is the share of validation images the model gets wrong.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(5)

## Step 3: Test the model on new photos

The classes are `sofa` and `table`. To make the code independent of class order, the probability of "sofa" is looked up by name.

In [ ]:
print('Classes:', learn.dls.vocab)

def test_image(file):
    label, _, probs = learn.predict(PILImage.create(file))
    p_sofa = probs[learn.dls.vocab.o2i['sofa']]
    print(f'{file}: this is a {label}. Probability it is a sofa: {p_sofa:.4f}')

# 1. The sofa photo downloaded at the start
test_image('sofa.jpg')

# 2. A different sofa
download_url('https://plus.unsplash.com/premium_photo-1681449856688-2abd99ab5a73?fm=jpg&q=60&w=3000&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=NHwxMjA3fDA%3D', 'sofa2.jpg', show_progress=False)
test_image('sofa2.jpg')

# 3. A table
download_url('https://images.affordableseating.net/dpr_1.0,f_auto,q_auto,w_2000,h_2000/afd/media/catalog/product/s/w/swpl-set-3048-na-mfb-3048-bl-30h-1_2.png', 'table1.jpg', show_progress=False)
test_image('table1.jpg')

## Results

| Test image | Model answer | Probability of "sofa" |
|---|---|---|
| sofa.jpg | sofa | 0.6929 |
| sofa2.jpg (different sofa) | sofa | 0.8252 |
| table1.jpg | table | 0.1165 |

Final validation error rate: **0.154** (about 85% accuracy). The dataset has only 66 images (33 per class), so the validation set has about 13 images and one wrong image changes the error rate by roughly 8 percentage points. All three test photos were classified correctly, but the model is not very confident: the probabilities are only about 0.7-0.9.

## Limitations and what I learned

- The dataset is small (33 images per class), much fewer than the up to 200 results the search can return. Getting more images, for example by combining several different search queries, would be the first thing to improve.
- The validation set is small, so the error rate changes a lot from run to run and should be read as a rough estimate.
- The model only knows two classes, so it must answer "sofa" or "table" even for a photo of a chair or a bed. A high probability does not mean the answer is correct.
- Only three test photos were checked by hand. All of them are clean, easy photos; photos with bad lighting or several pieces of furniture would be a harder test.
- The training images come from a search engine, so they may contain duplicates or wrong images.
- Main takeaway: fine-tuning a pretrained model gives usable results with very little data and a few seconds of training, but more (and cleaner) training images would be needed to make it more confident.